# Classificação de risco de não alfabetização

Comparação dos modelos e das referências na avaliação temporal de 2025, a partir dos resultados agregados da execução. Defina `FIAP_RUN_DIR` com a pasta que contém `metrics.json`, `protocol_lock.json` e `run_receipt.json`.

In [ ]:
import json
import os
from pathlib import Path
import pandas as pd

run_dir = Path(os.environ.get('FIAP_RUN_DIR', '../reports'))
metrics = json.loads((run_dir / 'metrics.json').read_text(encoding='utf-8'))
lock = json.loads((run_dir / 'protocol_lock.json').read_text(encoding='utf-8'))
receipt = json.loads((run_dir / 'run_receipt.json').read_text(encoding='utf-8'))
assert metrics['run_id'] == lock['run_id'] == receipt['run_id']
assert metrics['protocol_lock_hash'] == lock['lock_hash']
assert receipt['temporal_lock_verified'] and receipt['evaluated_artifact_reload_verified']
print('Run:', metrics['run_id'], '| status:', metrics['status'])
print('Fonte:', receipt['source_fingerprint'])


## Risco contextual e seleção

A classe positiva é **não alfabetizado**. O modelo utiliza contexto territorial compartilhado e não constitui diagnóstico individual. A seleção usa cinco divisões agrupadas de desenvolvimento em 2024; calibração sigmoid e limiar F2 usam a reserva desse ano. A avaliação de 2025 ocorre após essas escolhas.

Pesos oficiais entram no ajuste e nas métricas. Imputação e escala são não ponderadas e ajustadas dentro de cada divisão de treinamento.

In [ ]:
classification = metrics['classification']
print('Selecionado:', classification['selected_model'], '| limiar:', classification['threshold'])
display(pd.DataFrame([classification['population']]))
display(pd.DataFrame(classification['selection']['cv_results']))


## Avaliação temporal e referências

Average precision (AP) é o critério principal de seleção. As matrizes de confusão, a precisão, o recall e F2 dependem do limiar definido na reserva. As tabelas comparam prevalência, risco do contexto anterior e o modelo selecionado. Os intervalos por município descrevem variabilidade condicional aos dados, sem reproduzir integralmente o desenho amostral oficial.

In [ ]:
rows = [{'model': 'selected', **classification['test']}, *[{'model': name, **values} for name, values in classification['baselines'].items()]]
columns = ['model', 'average_precision', 'roc_auc', 'precision', 'recall', 'f2', 'brier']
display(pd.DataFrame(rows)[columns])
display(pd.DataFrame(classification['test']['calibration_curve']))
display(pd.DataFrame(classification['subgroups']))
print(json.dumps(classification['bootstrap'], indent=2, ensure_ascii=False))


## Influência dos atributos e vazamento de informação

SHAP descreve o estimador-base antes da calibração, na classe e escala indicadas. A permutação mede a queda de AP ponderada ao trocar atributos entre blocos município/rede. As duas análises mostram associações preditivas, sem estimar efeitos causais.

A experiência de vazamento mantém a mesma regressão logística e divisão de 2024, acrescentando o próprio alvo somente nessa comparação demonstrativa.

In [ ]:
interpretation = metrics['interpretation']
print(interpretation['shap']['output_scale'])
display(pd.DataFrame(interpretation['shap']['importance']))
display(pd.DataFrame(interpretation['permutation_importance']))
display(pd.DataFrame([interpretation['leakage_demo']]))
print('\n'.join(metrics['limitations']))


## Modelos avaliados e projeção

Os modelos avaliados em 2025 são preservados separadamente dos reajustes destinados a 2026. As métricas do teste pertencem aos primeiros; não constituem avaliação independente dos reajustes. Os arquivos de modelo contêm estimadores e metadados, sem registros individuais ou bases de explicação.

In [ ]:
display(pd.DataFrame.from_dict(metrics['artifacts'], orient='index'))
